# Paper 6 — Built-environment features and spatial GeoAI (Colab)

The two OpenStreetMap extracts (`new-york-260903.osm.pbf`, 496 MB; `illinois-260903.osm.pbf`,
358 MB) are too large to process comfortably on a laptop, so this notebook runs on Colab with a
high-RAM runtime. It produces two things the main analysis merges in:

1. `osm_tract_features.parquet` — built-environment measures per census tract
   (street and intersection density, building footprint coverage, land-use mix, POI counts).
   These operationalise the *physical opportunity structure* that Broken Windows treats as the
   setting in which disorder is read.
2. `gnn_results.json` — a spatial graph neural network that predicts tract-level repeat
   complaining from the tract adjacency graph, benchmarked against the tabular LightGBM model.

**What to upload to Google Drive** (folder `Paper6_colab/`):
`new-york-260903.osm.pbf`, `illinois-260903.osm.pbf`,
`nyc_tract_month.parquet`, `chi_tract_month.parquet`, `tract_geo.parquet`
(the last three come out of `Paper6_00_Prep_LocalData.ipynb`).

On this lab node you can skip Drive: the notebook now detects a local `DataPaper6/Open_Street_Map/`
tree and writes into `derived/`. GPU is used if one has enough free VRAM, otherwise CPU.

```bash
python scripts/run_job.py start geoai
python scripts/run_job.py status
```

Runtime → Change runtime type → **High-RAM** (Colab). Total run time is roughly 40–70 minutes.

## 1 — Environment

In [ ]:
import os, json, time, warnings, sys
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    get_ipython().system("pip -q install pyrosm geopandas pyarrow libpysal esda lightgbm 2>&1 | tail -2")
    BASE = Path("/content/drive/MyDrive/Paper6_colab")
    OUT = BASE / "out"
    PBF_DIR = BASE
    GPKG = BASE / "tracts_ny_il.gpkg"
    TRACT_GEO = BASE / "tract_geo.parquet"
    LGB_KW = {}
else:
    here = Path.cwd()
    PROJECT = Path(os.environ["PAPER6_ROOT"]) if os.environ.get("PAPER6_ROOT") else here
    if not (PROJECT / "DataPaper6").exists() and (here / "DataPaper6").exists():
        PROJECT = here
    BASE = PROJECT / "derived"
    OUT = PROJECT / "derived"
    PBF_DIR = PROJECT / "DataPaper6" / "Open_Street_Map"
    GPKG = PROJECT / "derived" / "_geo" / "tracts_ny_il.gpkg"
    TRACT_GEO = PROJECT / "derived" / "tract_geo.parquet"
    sys.path.insert(0, str(PROJECT / "scripts"))
    try:
        from paper6_runtime import describe_compute, apply_cuda_visible, cap_blas_threads
        cap_blas_threads()
        _DEV = describe_compute("torch")
        apply_cuda_visible(_DEV)
        LGB_KW = {"device": "cpu", "n_jobs": _DEV.get("n_threads", 8)}
        print("compute:", _DEV["reason"])
        print("LightGBM in this notebook stays on CPU; the GNN uses the GPU if one is free.")
    except Exception as e:
        LGB_KW = {"device": "cpu", "n_jobs": min(8, os.cpu_count() or 8)}
        print("device helper:", e)

OUT.mkdir(parents=True, exist_ok=True)
print("IN_COLAB =", IN_COLAB)
print("BASE    =", BASE)
print("PBF_DIR =", PBF_DIR)
print("files present:")
seen = set()
for folder in (BASE, PBF_DIR, OUT):
    if not folder.exists():
        continue
    for p in sorted(folder.glob("*")):
        if p.is_file() and p.name not in seen:
            seen.add(p.name)
            print(f"  {p.name:<48} {p.stat().st_size/1e6:>9.1f} MB")

In [ ]:
import psutil
print("RAM (GB):", round(psutil.virtual_memory().total/1e9, 1))
if psutil.virtual_memory().total < 20e9:
    print("WARNING: OSM extracts want ~20+ GB RAM. On Colab: Runtime → High-RAM.")
else:
    print("RAM is enough for the OSM extracts.")

## 2 — Tract geometry

We reuse the TIGER boundaries the local prep already resolved, so tract identifiers line up
exactly with the panels. If `tract_geo.parquet` is missing the cell falls back to downloading
the shapefiles directly.

In [ ]:
tg_path = TRACT_GEO
gpkg    = GPKG
if gpkg.exists():
    tracts = gpd.read_file(gpkg)[["GEOID", "geometry"]].to_crs(4326)
else:
    import requests, zipfile, io
    parts = []
    for fips in ("36", "17"):
        url = f"https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_{fips}_tract.zip"
        z = BASE / f"tl_2023_{fips}_tract.zip"
        if not z.exists():
            r = requests.get(url, timeout=600); z.write_bytes(r.content)
        parts.append(gpd.read_file(f"zip://{z}")[["GEOID", "geometry"]])
    tracts = pd.concat(parts, ignore_index=True)
    tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4269").to_crs(4326)

NYC_C = {"36005","36047","36061","36081","36085"}
tracts["city"] = np.where(tracts.GEOID.str[:5].isin(NYC_C), "NYC",
                  np.where(tracts.GEOID.str[:5].eq("17031"), "CHI", "other"))
study = tracts[tracts.city.isin(["NYC","CHI"])].copy()
study_m = study.to_crs(3857)          # metric CRS for areas and lengths
study_m["area_m2"] = study_m.area
print(study.city.value_counts())
study.head(2)

## 3 — Built-environment features from the OSM extracts

For each city we read the road network, buildings and points of interest from the `.pbf`,
clip to the study tracts, and aggregate. Reading is done per layer and the intermediate
GeoDataFrames are released immediately, because the New York extract alone expands to several
GB in memory.

In [ ]:
from pyrosm import OSM

POI_GROUPS = {
    "poi_food_drink":  {"amenity": ["bar","pub","nightclub","restaurant","fast_food","cafe"]},
    "poi_alcohol":     {"shop": ["alcohol","wine"]},
    "poi_civic":       {"amenity": ["library","community_centre","school","place_of_worship",
                                    "townhall","social_facility"]},
    "poi_transit":     {"highway": ["bus_stop"], "railway": ["station","subway_entrance"]},
    "poi_vacant_sign": {"building": ["ruins"], "disused": ["yes"]},
}

def city_features(pbf_path, tracts_city, label):
    t0 = time.time()
    osm = OSM(str(pbf_path))
    tm = tracts_city.to_crs(3857).copy()
    tm["area_m2"] = tm.geometry.area
    feats = tm[["GEOID","area_m2"]].copy().set_index("GEOID")

    # --- street network -------------------------------------------------------
    net = osm.get_network(network_type="driving")
    net = gpd.GeoDataFrame(net, geometry="geometry", crs=4326).to_crs(3857)
    net = net[net.geometry.notna()]
    j = gpd.sjoin(net[["geometry"]], tm[["GEOID","geometry"]], how="inner", predicate="intersects")
    j["len_m"] = j.geometry.length
    feats["street_len_m"] = j.groupby("GEOID").len_m.sum()
    # intersection density from shared endpoints
    ends = []
    for geom in net.geometry.head(400_000):
        if geom is None: continue
        try:
            cs = list(geom.coords); ends.append(cs[0]); ends.append(cs[-1])
        except NotImplementedError:
            pass
    ep = pd.DataFrame(ends, columns=["x","y"]).round(0)
    node_deg = ep.groupby(["x","y"]).size().rename("deg").reset_index()
    nodes = gpd.GeoDataFrame(node_deg[node_deg.deg >= 3],
                             geometry=gpd.points_from_xy(node_deg.loc[node_deg.deg>=3,"x"],
                                                         node_deg.loc[node_deg.deg>=3,"y"]),
                             crs=3857)
    jn = gpd.sjoin(nodes, tm[["GEOID","geometry"]], how="inner", predicate="within")
    feats["n_intersections"] = jn.groupby("GEOID").size()
    del net, j, ends, ep, node_deg, nodes, jn
    print(f"  [{label}] network done {time.time()-t0:.0f}s")

    # --- buildings ------------------------------------------------------------
    b = osm.get_buildings()
    if b is not None and len(b):
        b = gpd.GeoDataFrame(b, geometry="geometry", crs=4326).to_crs(3857)
        b = b[b.geometry.notna() & b.geometry.is_valid]
        b["fp"] = b.geometry.area
        jb = gpd.sjoin(b[["fp","geometry"]], tm[["GEOID","geometry"]],
                       how="inner", predicate="intersects")
        feats["n_buildings"]  = jb.groupby("GEOID").size()
        feats["building_m2"]  = jb.groupby("GEOID").fp.sum()
        del b, jb
    print(f"  [{label}] buildings done {time.time()-t0:.0f}s")

    # --- land use mix ---------------------------------------------------------
    lu = osm.get_landuse()
    if lu is not None and len(lu):
        lu = gpd.GeoDataFrame(lu, geometry="geometry", crs=4326).to_crs(3857)
        lu = lu[lu.geometry.notna() & lu.geometry.is_valid]
        lu["a"] = lu.geometry.area
        jl = gpd.sjoin(lu[["landuse","a","geometry"]], tm[["GEOID","geometry"]],
                       how="inner", predicate="intersects")
        piv = jl.pivot_table(index="GEOID", columns="landuse", values="a", aggfunc="sum").fillna(0)
        sh = piv.div(piv.sum(axis=1).replace(0, np.nan), axis=0)
        feats["landuse_entropy"] = -(sh * np.log(sh.replace(0, np.nan))).sum(axis=1)
        for c in ["residential","commercial","industrial","retail"]:
            if c in sh.columns: feats[f"lu_{c}"] = sh[c]
        del lu, jl, piv, sh
    print(f"  [{label}] landuse done {time.time()-t0:.0f}s")

    # --- points of interest ---------------------------------------------------
    for name, flt in POI_GROUPS.items():
        try:
            p = osm.get_pois(custom_filter=flt)
        except Exception:
            p = None
        if p is None or len(p) == 0:
            feats[name] = 0; continue
        p = gpd.GeoDataFrame(p, geometry="geometry", crs=4326).to_crs(3857)
        p["geometry"] = p.geometry.centroid
        jp = gpd.sjoin(p[["geometry"]], tm[["GEOID","geometry"]], how="inner", predicate="within")
        feats[name] = jp.groupby("GEOID").size()
        del p, jp
    print(f"  [{label}] POIs done {time.time()-t0:.0f}s")

    feats = feats.fillna(0)
    km2 = feats["area_m2"] / 1e6
    feats["street_km_per_km2"]   = feats.street_len_m / 1000 / km2
    feats["intersect_per_km2"]   = feats.n_intersections / km2
    feats["building_coverage"]   = feats.building_m2 / feats.area_m2
    feats["buildings_per_km2"]   = feats.n_buildings / km2
    for name in POI_GROUPS:
        feats[f"{name}_per_km2"] = feats[name] / km2
    feats["city"] = label
    del osm
    import gc; gc.collect()
    return feats.reset_index()

frames = []
jobs = [("illinois-260903.osm.pbf", "CHI")]
if (BASE / "nyc_tract_month.parquet").exists():
    jobs.insert(0, ("new-york-260903.osm.pbf", "NYC"))
else:
    print("NYC panel missing — OSM extract for Illinois/Chicago only")
for pbf, city in jobs:
    p = PBF_DIR / pbf
    if not p.exists():
        print("missing", pbf, "- skipped"); continue
    sub = study[study.city == city]
    if sub.empty:
        print("no tracts for", city, "- skipped"); continue
    frames.append(city_features(p, sub, city))
osm_feat = pd.concat(frames, ignore_index=True)
osm_feat.to_parquet(OUT / "osm_tract_features.parquet", index=False)
print(osm_feat.shape)
osm_feat.describe().T.round(3)

## 4 — Do built-environment features add explanatory power?

We merge the OSM measures onto the tract-month panel and re-fit the LightGBM model with and
without them. The comparison is on the same held-out final 12 months used in the main analysis,
so the increment is directly interpretable.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import r2_score

_panel = BASE / "nyc_tract_month.parquet"
CITY_TAG = "NYC"
if not _panel.exists():
    _panel = BASE / "chi_tract_month.parquet"
    CITY_TAG = "CHI"
    print("using Chicago panel + Illinois OSM features")
nyc = pd.read_parquet(_panel)
nyc["month"] = pd.to_datetime(nyc.month)
nyc["t"] = (nyc.month.dt.year - nyc.month.dt.year.min())*12 + nyc.month.dt.month - 1
nyc = nyc[(nyc["pop"] > 200) & (nyc.n_complaints >= 3)].sort_values(["GEOID","t"])
for col in ["n_physical_disorder", "n_social_disorder"]:
    if col not in nyc.columns:
        nyc[col] = 0
nyc["disorder_rate"] = 1000*(nyc.n_physical_disorder + nyc.n_social_disorder)/nyc["pop"]
nyc["complaint_rate"] = 1000*nyc.n_complaints/nyc["pop"]
for col in ["disorder_rate","complaint_rate","repeat_rate","median_days_close","share_closed_3d"]:
    nyc[col+"_l1"] = nyc.groupby("GEOID")[col].shift(1)
osm_city = CITY_TAG if "city" in osm_feat.columns and (osm_feat.city == CITY_TAG).any() else osm_feat.city.iloc[0]
print("merge OSM city =", osm_city)
nyc = nyc.merge(osm_feat[osm_feat.city.eq(osm_city)] if "city" in osm_feat.columns else osm_feat,
                on="GEOID", how="left")

BASE_F = ["disorder_rate_l1","complaint_rate_l1","repeat_rate_l1","median_days_close_l1",
          "share_closed_3d_l1","pct_vacant","pct_owner_occ","pct_unemployed","pct_ba_plus","t"]
OSM_F  = ["street_km_per_km2","intersect_per_km2","building_coverage","buildings_per_km2",
          "landuse_entropy","poi_food_drink_per_km2","poi_alcohol_per_km2",
          "poi_civic_per_km2","poi_transit_per_km2"]
BASE_F = [c for c in BASE_F if c in nyc.columns]
OSM_F  = [c for c in OSM_F  if c in nyc.columns]

d = nyc.dropna(subset=["repeat_rate"] + BASE_F).copy()
cut = d.t.max() - 12
tr, te = d[d.t <= cut], d[d.t > cut]
res = {}
for name, feats in [("without OSM", BASE_F), ("with OSM", BASE_F + OSM_F)]:
    m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=63,
                          min_child_samples=60, subsample=0.85, subsample_freq=1,
                          colsample_bytree=0.8, random_state=7, verbose=-1, **LGB_KW)
    m.fit(tr[feats], tr.repeat_rate)
    res[name] = float(r2_score(te.repeat_rate, m.predict(te[feats])))
    if name == "with OSM":
        gain = pd.DataFrame({"feature": feats, "gain": m.booster_.feature_importance("gain")}) \
                 .sort_values("gain", ascending=False)
print(json.dumps(res, indent=1))
gain.head(15)

## 5 — Spatial graph neural network

Tracts form a graph: nodes are tracts, edges join tracts that share a boundary. A two-layer
GraphSAGE model propagates neighbourhood information along that graph, which is the structural
counterpart of the spatial-lag specification in the main analysis. We predict the tract's mean
repeat-complaining rate and compare against the tabular benchmark.

In [ ]:
try:
    import torch
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torch-geometric"])
    import torch
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv
import torch.nn.functional as Fn
from libpysal.weights import Queen

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch device:", device, flush=True)

node = (nyc.groupby("GEOID")
          .agg(y=("repeat_rate","mean"), **{c: (c,"mean") for c in BASE_F + OSM_F if c != "t"})
          .dropna().reset_index())
CITY_TAG = globals().get("CITY_TAG", "CHI")
geo = study[study.city.eq(CITY_TAG)].merge(node[["GEOID"]], on="GEOID")
geo = geo.reset_index(drop=True)
node = node.set_index("GEOID").loc[geo.GEOID].reset_index()

w = Queen.from_dataframe(geo, use_index=False)
edges = [[i, j] for i, ns in w.neighbors.items() for j in ns]
edge_index = torch.tensor(np.array(edges).T, dtype=torch.long)
Xn = node.drop(columns=["GEOID","y"]).values.astype("float32")
Xn = (Xn - Xn.mean(0)) / (Xn.std(0) + 1e-8)
yn = node.y.values.astype("float32")

rng = np.random.default_rng(7); idx = rng.permutation(len(node))
ntr = int(0.7*len(idx)); nva = int(0.85*len(idx))
masks = {}
for nm, sl in [("train", idx[:ntr]), ("val", idx[ntr:nva]), ("test", idx[nva:])]:
    m = torch.zeros(len(node), dtype=torch.bool); m[sl] = True; masks[nm] = m.to(device)
data = Data(x=torch.tensor(Xn), y=torch.tensor(yn), edge_index=edge_index).to(device)

class SAGE(torch.nn.Module):
    def __init__(self, d_in, h=64):
        super().__init__()
        self.c1, self.c2 = SAGEConv(d_in, h), SAGEConv(h, h//2)
        self.out = torch.nn.Linear(h//2, 1)
    def forward(self, x, ei):
        x = Fn.relu(self.c1(x, ei)); x = Fn.dropout(x, 0.2, self.training)
        x = Fn.relu(self.c2(x, ei))
        return self.out(x).squeeze(-1)

model = SAGE(Xn.shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
best, best_state, patience = 1e9, None, 0
for ep in range(600):
    model.train(); opt.zero_grad()
    pred = model(data.x, data.edge_index)
    loss = Fn.mse_loss(pred[masks["train"]], data.y[masks["train"]])
    loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        vl = Fn.mse_loss(model(data.x, data.edge_index)[masks["val"]], data.y[masks["val"]]).item()
    if vl < best - 1e-7:
        best, best_state, patience = vl, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
    else:
        patience += 1
        if patience > 60: break
model.load_state_dict(best_state); model.to(device).eval()
with torch.no_grad():
    p = model(data.x, data.edge_index).detach().cpu().numpy()
from sklearn.metrics import r2_score as R2
te_m = masks["test"].detach().cpu().numpy()
gnn_r2 = float(R2(yn[te_m], p[te_m]))

mlp_bench = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1, random_state=1, **LGB_KW)
mlp_bench.fit(Xn[masks["train"].detach().cpu().numpy()], yn[masks["train"].detach().cpu().numpy()])
tab_r2 = float(R2(yn[te_m], mlp_bench.predict(Xn[te_m])))

out = {"gnn_test_r2": gnn_r2, "tabular_test_r2": tab_r2,
       "lgbm_panel_r2": res, "n_nodes": int(len(node)), "n_edges": int(edge_index.shape[1])}
(OUT / "gnn_results.json").write_text(json.dumps(out, indent=1))
print(json.dumps(out, indent=1))

## 6 — Bring the results home

## 6 — Bring the results home

On the lab machine the parquet/json already land in `derived/`. On Colab, download
`out/osm_tract_features.parquet` and `out/gnn_results.json` from Drive into
`Paper6/derived/`. The main analysis notebook picks them up automatically and reports the
built-environment increment and the GNN benchmark in the robustness section.

In [ ]:
for p in sorted(OUT.glob("*")):
    if p.suffix in {".parquet", ".json"} or "osm" in p.name or "gnn" in p.name:
        print(f"{p.name:<36} {p.stat().st_size/1e6:>8.2f} MB")
print("\nOutputs are in", OUT)